In [54]:
import pandas as pd
import os
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
from PIL import Image
from openai import OpenAI

In [3]:
artnet_2025_cover = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet 2025-09-03.xlsx",sheet_name=0)

In [2]:
artnet_2025_data1 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet 2025-09-03.xlsx",sheet_name=1)

In [3]:
artnet_2025_data2 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet 2025-09-03.xlsx",sheet_name=2)

# Valid Image and "Sold"

Limited to artwork that are sold.

In [4]:
image_files = os.listdir("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images")

In [5]:
artnet_2025 = pd.concat([artnet_2025_data1,artnet_2025_data2], ignore_index=True)

In [6]:
image_files_set = set(image_files)
filenames = artnet_2025['artwork id'].astype(str).add('.jpg')

In [7]:
artnet_2025['has_image'] = filenames.map(image_files_set.__contains__).astype('uint8')

In [8]:
artnet_2025_sold = artnet_2025[artnet_2025["pricePhrase"]=="Sold"]

In [9]:
artnet_2025_sold_image = artnet_2025_sold[artnet_2025_sold['has_image']==1]

In [10]:
artnet_2025_sold_image.head()

,lot id,artwork id,artist id,sale date,artist modifier,first,last,nationality,year born,year died,...,sale price,currency,currency exchange rate,est lo usd,est hi usd,sale price usd,priceStatus,pricePhrase,artnet lot image url,has_image
0,440090466,3,17524,2021-05-13 00:00:00,NaN,Andy,Warhol,American,1928.0,1987.0,...,478800.0,US Dollar,1.0000,3.000000e+05,5.000000e+05,4.788000e+05,Premium,Sold,https://www.artnet.com/WebServices/picture.asp...,1
2,428991578,12,7116,2012-02-07 00:00:00,NaN,Vincent van,Gogh,Dutch,1853.0,1890.0,...,10121250.0,British Pound,0.6294,7.944074e+06,1.112170e+07,1.608079e+07,Premium,Sold,https://www.artnet.com/WebServices/picture.asp...,1
3,436942346,12,7116,2018-05-15 00:00:00,NaN,Vincent van,Gogh,Dutch,1853.0,1890.0,...,39687500.0,US Dollar,1.0000,3.500000e+07,5.500000e+07,3.968750e+07,Premium,Sold,https://www.artnet.com/WebServices/picture.asp...,1
4,425918584,16,19596,2008-12-15 14:00:00,NaN,Serge,Charchoune,French/Russian,1888.0,1975.0,...,5500.0,Euro,0.7440,3.360215e+03,4.704301e+03,7.392473e+03,Hammer,Sold,https://www.artnet.com/WebServices/picture.asp...,1
5,440138106,16,19596,2021-05-26 00:00:00,NaN,Serge,Charchoune,French/Russian,1888.0,1975.0,...,3500.0,British Pound,0.7080,2.824859e+03,4.237288e+03,4.943503e+03,Hammer,Sold,https://www.artnet.com/WebServices/picture.asp...,1


In [12]:
artnet_2025_sold_image= artnet_2025_sold_image.drop("artnet lot image url", axis=1)

In [13]:
artnet_2025_sold_image.to_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_sold.xlsx",index=False)

## Valid Year and Year Ordered

Remove artwork without create time

In [3]:
artnet_2025= pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_image_cleaned.xlsx")

In [8]:
artnet_2025_embed = np.load("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\clip_embedding.npy",allow_pickle=True)

In [4]:
artnet_2025.head()

,lot id,artwork id,artist id,sale date,artist modifier,first,last,nationality,year born,year died,...,est hi,sale price,currency,currency exchange rate,est lo usd,est hi usd,sale price usd,priceStatus,pricePhrase,has_image
0,440090466,3,17524,2021-05-13 00:00:00,NaN,Andy,Warhol,American,1928.0,1987.0,...,500000.0,478800.0,US Dollar,1.0000,3.000000e+05,5.000000e+05,4.788000e+05,Premium,Sold,1
1,428991578,12,7116,2012-02-07 00:00:00,NaN,Vincent van,Gogh,Dutch,1853.0,1890.0,...,7000000.0,10121250.0,British Pound,0.6294,7.944074e+06,1.112170e+07,1.608079e+07,Premium,Sold,1
2,436942346,12,7116,2018-05-15 00:00:00,NaN,Vincent van,Gogh,Dutch,1853.0,1890.0,...,55000000.0,39687500.0,US Dollar,1.0000,3.500000e+07,5.500000e+07,3.968750e+07,Premium,Sold,1
3,425918584,16,19596,2008-12-15 14:00:00,NaN,Serge,Charchoune,French/Russian,1888.0,1975.0,...,3500.0,5500.0,Euro,0.7440,3.360215e+03,4.704301e+03,7.392473e+03,Hammer,Sold,1
4,440138106,16,19596,2021-05-26 00:00:00,NaN,Serge,Charchoune,French/Russian,1888.0,1975.0,...,3000.0,3500.0,British Pound,0.7080,2.824859e+03,4.237288e+03,4.943503e+03,Hammer,Sold,1


In [51]:
artnet_2025.columns

Index(['lot id', 'artwork id', 'artist id', 'sale date', 'artist modifier',
       'first', 'last', 'nationality', 'year born', 'year died', 'title',
       'workyear modifier', 'workyear from', 'workyear to', 'est lo', 'est hi',
       'sale price', 'currency', 'currency exchange rate', 'est lo usd',
       'est hi usd', 'sale price usd', 'priceStatus', 'pricePhrase',
       'has_image', 'sale year'],
      dtype='object')

In [18]:
artnet_2025_year = artnet_2025.dropna(subset=['workyear from'])

In [19]:
artnet_2025_year.shape

(530853, 26)

In [20]:
artnet_2025_year_reorder = artnet_2025_year.sort_values(by='workyear from')

In [21]:
artnet_2025_year_reorder.to_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_year_sorted.xlsx",index=False)

# Artist Check

Nationality is checked with ChatGPT

In [50]:
with open("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Token_Key\\api.txt", "r", encoding="utf-8") as file:
    api = file.read()

In [55]:
client = OpenAI(api_key=api)

In [56]:
Europe_list =[]
for i in artnet_2025_year_reorder["nationality"].unique():
    message = [{'role':'user', 'content':f"Does {i} part of Europe? Please reply only with 'Yes' or 'No'"}]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=message,
        temperature=0.7,
        seed=42
    )
    answer = response.choices[0].message.content
    if answer.lower()=='yes':
        Europe_list.append(i)

In [ ]:
Europe_list

In [60]:
artnet_2025_artist = artnet_2025_year_reorder[artnet_2025_year_reorder["nationality"].isin(Europe_list)]

In [64]:
artnet_2025_year_reorder["Europe"]=artnet_2025_year_reorder["nationality"].isin(Europe_list).astype(int)

In [65]:
artnet_2025_year_reorder.to_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_year_sorted.xlsx",index=False)

In [66]:
artnet_2025_artist.to_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe.xlsx",index=False)

## Valid Year Range

remove artwork created before 500 A.D., because Medieval started from 500 to 1400.

In [67]:
artnet_2025_artist[['artwork id','workyear from']]

,artwork id,workyear from
408762,434647885,6.0
507472,439702536,189.0
523762,440146555,196.0
523457,440134531,960.0
388708,434101355,1076.0
...,...,...
608657,442469320,2022.0
622226,442935377,2022.0
634685,443255015,2022.0
649988,443629756,2023.0


In [71]:
artnet_2025_artist = artnet_2025_artist[(artnet_2025_artist['workyear from'] > 500) & (artnet_2025_artist['workyear from'] <= 2025)]

In [72]:
artnet_2025_artist.to_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe.xlsx",index=False)

In [ ]:
artwork_id = 432825214	

In [ ]:
img = Image.open(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\{434101355}.jpg")
img.show()  # Opens with default viewer (outside notebook)

# Artwork Check

In [92]:
a = artnet_2025_artist["artwork id"].value_counts()

In [96]:
a

artwork id
6652         5
746          5
10714        5
17641        5
2369         5
            ..
430447694    1
441326034    1
434136558    1
440722158    1
433828582    1
Name: count, Length: 368440, dtype: int64

In [93]:
num_larger_than_one = (a > 1).sum()

In [94]:
num_larger_than_one

np.int64(2814)

In [95]:
artnet_2025_artist["artwork id"].unique()

array([432409847, 427653432, 430117545, ..., 434136558, 440722158,
       444870863], shape=(368440,))

# Create Year Valid Check

In [80]:
artnet_2025_artist[['workyear from','year born','year died']]

,workyear from,year born,year died
523457,960.0,1904.0,1989.0
388708,1076.0,1888.0,1967.0
386668,1095.0,1861.0,1927.0
406093,1191.0,1833.0,1898.0
560747,1400.0,1471.0,1528.0
...,...,...,...
610777,2022.0,1928.0,1962.0
608657,2022.0,1928.0,1962.0
622226,2022.0,1928.0,1962.0
634685,2022.0,1929.0,2005.0


make sure the create year of artwork fill within the lifetime of artist

In [84]:
artnet_2025_artist = artnet_2025_artist[artnet_2025_artist['workyear from'] > artnet_2025_artist['year born']]

In [89]:
artnet_2025_artist = artnet_2025_artist[artnet_2025_artist['workyear from'] <= artnet_2025_artist['year died']]

In [90]:
artnet_2025_artist.to_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe.xlsx",index=False)

In [91]:
artnet_2025_artist.shape

(371778, 26)

# Prior Knowledge

In [99]:
artnet_2025_artist[artnet_2025_artist['workyear from']<=1600]

,lot id,artwork id,artist id,sale date,artist modifier,first,last,nationality,year born,year died,...,sale price,currency,currency exchange rate,est lo usd,est hi usd,sale price usd,priceStatus,pricePhrase,has_image,sale year
338545,432409847,432409847,593645,2014-02-27 00:00:00,Circle Of,Rogier van der,Weyden,Flemish,1399.0,1464.0,...,40000.0,Euro,0.7293,13711.778418,13711.778418,54847.113671,Hammer,Sold,1,2014
208309,427653432,427653432,662454,2010-12-15 14:00:00,Workshop Of,Francesco di Gentile da,Fabriano,Italian,1370.0,1427.0,...,34720.0,Euro,0.7515,39920.159681,53226.879574,46200.931470,Premium,Sold,1,2010
278481,430117545,430117545,643948,2012-10-27 00:00:00,NaN,Fra,Angelico,Italian,1400.0,1455.0,...,445000.0,Euro,0.7727,NaN,NaN,575902.678918,Hammer,Sold,1,2012
401785,434483742,434483742,553790,2015-11-14 00:00:00,Follower Of,Hans,Memling,German,1430.0,1494.0,...,334800.0,Euro,0.9289,86123.371730,107654.214663,360426.310690,Premium,Sold,1,2015
539285,440596343,440596343,15130,2021-10-23 00:00:00,NaN,Martin,Schongauer,German,1445.0,1491.0,...,4200.0,Euro,0.8589,1397.135871,1397.135871,4889.975550,Hammer,Sold,1,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472856,438598564,438598564,555274,2019-12-12 00:00:00,Follower Of,NaN,Guercino,Italian,1591.0,1666.0,...,310000.0,Swedish Krona,9.4064,10631.059704,15946.589556,32956.285082,Hammer,Sold,1,2019
344290,432619228,432619228,13707,2014-04-24 00:00:00,Follower Of,Nicolas,Poussin,French,1594.0,1665.0,...,6500.0,Euro,0.7232,3456.858407,4839.601770,8987.831858,Hammer,Sold,1,2014
314723,431543521,431543521,642916,2013-06-14 00:00:00,School Of,NaN,Caravaggio,Italian,1571.0,1610.0,...,14500.0,Euro,0.7501,7998.933476,9332.089055,19330.755899,Hammer,Sold,1,2013
321575,431854628,431854628,642916,2013-10-15 00:00:00,Follower Of,NaN,Caravaggio,Italian,1571.0,1610.0,...,91800.0,Euro,0.7411,80960.734044,107947.645392,123869.923087,Premium,Sold,1,2013
